# 🩻 Clinical Pathology Classification Head & CheXpert Evaluation
### Rule-Based Clinical Label Extraction & Multi-Label Diagnostic Evaluation for NaijaCXR-VLM

This notebook implements an automated clinical classification evaluation pipeline based on the **CheXpert labeler methodology**. It maps unstructured generated radiology text (Findings & Impression) and Ground Truth reports into standardized clinical pathology vectors, computing multi-label diagnostic accuracy, F1-scores, and clinical concordance.

---

### Key Pathologies Evaluated:
1. **Cardiomegaly**
2. **Edema / Vascular Congestion**
3. **Consolidation / Pneumonia**
4. **Pleural Effusion**
5. **Pneumothorax**
6. **TB / Reticulonodular Opacities** (Tailored for African clinical settings)
7. **Atelectasis / Collapse**
8. **No Finding / Normal Study**


## 1. Environment Setup & Dependencies


In [ ]:
import os
import sys
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    multilabel_confusion_matrix,
    hamming_loss,
    jaccard_score
)
from sklearn.preprocessing import MultiLabelBinarizer

# Set visualization style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10

# Mount Google Drive if in Google Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')


## 2. Configuration & Prediction File Ingestion


In [ ]:
# =========================================================
# CONFIGURATION & PATH RESOLUTION
# =========================================================
# Auto-detect JSON or CSV predictions
candidate_paths = [
    "/content/naija_predictions.json",
    "/content/NaijaCXR-VLM_predictions.csv",
    "/content/drive/MyDrive/NaijaCXR_Project/naija_predictions.json",
    "/content/drive/MyDrive/NaijaCXR_Project/NaijaCXR-VLM_predictions.csv",
    "naija_predictions.json",
    "NaijaCXR-VLM_predictions.csv",
    "MIMIC_CXR_NaijaCXR_VLM_predictions.csv"
]

PRED_FILE = next((p for p in candidate_paths if os.path.exists(p)), "naija_predictions.json")
OUTPUT_METRICS_CSV = "clinical_classification_metrics.csv"
OUTPUT_METRICS_JSON = "clinical_classification_metrics.json"

print(f"📁 Target Predictions File: {PRED_FILE} (Found: {os.path.exists(PRED_FILE)})")


## 3. Clinical Pathology Lexicon & Negation Rules
We define comprehensive clinical keyword dictionaries along with negation detection patterns (`"no evidence of"`, `"clear"`, `"free"`, `"unremarkable"`).


In [ ]:
# ---------------------------------------------------------
# 1. DEFINE CLINICAL LABELS & PATHOLOGY PATTERNS
# ---------------------------------------------------------
CHEXPERT_PATTERNS = {
    "Cardiomegaly": [
        "cardiomegaly", "enlarged heart", "heart size is enlarged",
        "cardiac silhouette is enlarged", "multichamber configuration",
        "left ventricular configuration", "ctr >", "ctr is 5", "ctr is 6", "ctr is 7",
        "ctr is 8", "cardiac enlargement", "enlarged cardiac silhouette"
    ],
    "Edema": [
        "edema", "oedema", "vascular congestion", "cephalization",
        "upper lobe diversion", "engorgement", "vascular prominence",
        "fluid overload", "perihilar haze", "batwing opacity"
    ],
    "Consolidation": [
        "consolidation", "air bronchogram", "dense opacity", "alveolar opacity",
        "air-space opacity", "pneumonia", "bronchopneumonia", "lobar consolidation"
    ],
    "Pleural Effusion": [
        "effusion", "pleural effusion", "costophrenic angle", "costophrenic sulcus",
        "blunting", "meniscus", "fluid level", "hydro-pneumothorax", "fluid in pleural space"
    ],
    "Pneumothorax": [
        "pneumothorax", "collapsed lung", "pleural line", "air in pleural space",
        "tension pneumothorax", "apical pneumothorax"
    ],
    "TB/Opacities": [
        "tuberculosis", "ptb", "reticulonodular", "cavitation", "cavitary lesion",
        "fibrosis", "opacity", "opacities", "hazy", "haziness", "infiltrate",
        "patchy opacity", "streaky opacity", "micronodular"
    ],
    "Atelectasis": [
        "atelectasis", "volume loss", "collapse", "discoid atelectasis",
        "linear atelectasis", "subsegmental atelectasis"
    ]
}

# Negation & Normal triggers
NEGATION_TRIGGERS = [
    "no ", "not ", "without ", "free ", "clear ", "negative for ",
    "rules out ", "unremarkable ", "no evidence of ", "no sign of ",
    "absence of ", "intact ", "normal "
]


def get_labels(text):
    """
    Scans radiology text for pathology keywords with contextual negation detection.
    Returns a list of detected clinical pathologies, or ['No Finding'].
    """
    if not isinstance(text, str) or not text.strip():
        return ["No Finding"]

    text_lower = text.lower()
    detected = []
    found_any = False

    for pathology, keywords in CHEXPERT_PATTERNS.items():
        matched = False
        for k in keywords:
            # Look for keyword match
            start_pos = 0
            while True:
                idx = text_lower.find(k, start_pos)
                if idx == -1:
                    break

                # Extract preceding context window (up to 30 characters before keyword)
                context_start = max(0, idx - 30)
                context_window = text_lower[context_start:idx]

                # Check if negated
                is_negated = any(neg in context_window for neg in NEGATION_TRIGGERS)

                # Special medical phrase checking
                # e.g., "costophrenic sulci are free" -> free means NO effusion
                if pathology == "Pleural Effusion" and ("free" in text_lower[idx:idx+35] or "clear" in text_lower[idx:idx+35]):
                    is_negated = True

                if not is_negated:
                    detected.append(pathology)
                    found_any = True
                    matched = True
                    break

                start_pos = idx + len(k)

            if matched:
                break

    # If no specific disease was triggered, check for normal / no finding
    if not found_any:
        if any(norm in text_lower for norm in ["normal", "unremarkable", "clear", "intact", "no acute", "within normal"]):
            detected.append("No Finding")
        else:
            detected.append("No Finding")

    return list(set(detected))


## 4. Ingestion & Rule-Based Label Extraction


In [ ]:
# Load dataset from JSON or CSV
print(f"📖 Ingesting reports from: {PRED_FILE}")

if PRED_FILE.endswith(".json"):
    with open(PRED_FILE, 'r') as f:
        data = json.load(f)
    samples = []
    for item in data:
        gt = item.get("ground_truth", "")
        pred = item.get("prediction", "")
        img = item.get("image_path", item.get("image", item.get("id", "")))
        samples.append({"image": img, "ground_truth": gt, "prediction": pred})
    df_raw = pd.DataFrame(samples)
else:
    df_raw = pd.read_csv(PRED_FILE)

print(f"✅ Loaded {len(df_raw)} report pairs for clinical evaluation.")

# Extract labels for ground truth and predictions
y_true_labels = []
y_pred_labels = []

for idx, row in df_raw.iterrows():
    gt_text = str(row.get("ground_truth", ""))
    pred_text = str(row.get("prediction", ""))

    gt_lbls = get_labels(gt_text)
    pred_lbls = get_labels(pred_text)

    y_true_labels.append(gt_lbls)
    y_pred_labels.append(pred_lbls)

df_raw["gt_labels"] = y_true_labels
df_raw["pred_labels"] = y_pred_labels

print(f"\nSample Extracted Labels (Case 0):")
print(f"  Ground Truth: {df_raw['gt_labels'].iloc[0]}")
print(f"  Predicted:    {df_raw['pred_labels'].iloc[0]}")


## 5. Multi-Label Binarization


In [ ]:
# Fit MultiLabelBinarizer
mlb = MultiLabelBinarizer()
y_true = mlb.fit_transform(y_true_labels)
y_pred = mlb.transform(y_pred_labels)

classes = list(mlb.classes_)
num_classes = len(classes)

print(f"🏷️ Clinical Classes ({num_classes}): {classes}")
print(f"📊 Encoded Ground Truth Matrix Shape: {y_true.shape}")
print(f"📊 Encoded Prediction Matrix Shape:   {y_pred.shape}")


## 6. Multi-Label Diagnostic Evaluation Metrics


In [ ]:
# 1. Global Multi-Label Performance Metrics
exact_match_acc = accuracy_score(y_true, y_pred)
h_loss = hamming_loss(y_true, y_pred)
hamming_score_val = 1.0 - h_loss
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
micro_prec = precision_score(y_true, y_pred, average="micro", zero_division=0)
macro_prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
micro_rec = recall_score(y_true, y_pred, average="micro", zero_division=0)
macro_rec = recall_score(y_true, y_pred, average="macro", zero_division=0)

global_summary = {
    "Exact Match (Subset) Accuracy": round(exact_match_acc * 100, 2),
    "Hamming Score (Accuracy)": round(hamming_score_val * 100, 2),
    "Hamming Loss": round(h_loss, 4),
    "Micro Precision": round(micro_prec, 4),
    "Micro Recall": round(micro_rec, 4),
    "Micro F1-Score": round(micro_f1, 4),
    "Macro Precision": round(macro_prec, 4),
    "Macro Recall": round(macro_rec, 4),
    "Macro F1-Score": round(macro_f1, 4),
    "Weighted F1-Score": round(weighted_f1, 4)
}

print("=" * 65)
print("       🏥 GLOBAL CLINICAL CLASSIFICATION PERFORMANCE")
print("=" * 65)
for k, v in global_summary.items():
    print(f"  {k:32s}: {v}")
print("=" * 65)


## 7. Per-Class Diagnostic Performance Breakdown


In [ ]:
# Compute detailed per-class diagnostics
per_class_metrics = []
conf_matrices = multilabel_confusion_matrix(y_true, y_pred)

for i, cls_name in enumerate(classes):
    tn, fp, fn, tp = conf_matrices[i].ravel()
    
    cls_prec = precision_score(y_true[:, i], y_pred[:, i], zero_division=0)
    cls_rec = recall_score(y_true[:, i], y_pred[:, i], zero_division=0)
    cls_f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
    cls_acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gt_support = int(tp + fn)
    pred_support = int(tp + fp)

    per_class_metrics.append({
        "Pathology": cls_name,
        "Precision": round(cls_prec, 4),
        "Recall (Sensitivity)": round(cls_rec, 4),
        "Specificity": round(specificity, 4),
        "F1-Score": round(cls_f1, 4),
        "Accuracy": round(cls_acc, 4),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "GT Count": gt_support,
        "Pred Count": pred_support
    })

df_per_class = pd.DataFrame(per_class_metrics)

# Display formatted table
print("\n" + "=" * 90)
print("                      📋 PER-PATHOLOGY DIAGNOSTIC BREAKDOWN")
print("=" * 90)
print(df_per_class.to_string(index=False))
print("=" * 90)

# Save to CSV and JSON
df_per_class.to_csv(OUTPUT_METRICS_CSV, index=False)
with open(OUTPUT_METRICS_JSON, "w") as f:
    json.dump({
        "global_metrics": global_summary,
        "per_class_metrics": per_class_metrics
    }, f, indent=2)

print(f"\n💾 Exported metrics to: {OUTPUT_METRICS_CSV} and {OUTPUT_METRICS_JSON}")


## 8. Clinical Diagnostic Visualizations


In [ ]:
# 1. Pathology Prevalence Comparison Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df_per_class))
width = 0.35

rects1 = ax.bar(x - width/2, df_per_class["GT Count"], width, label="Ground Truth Prevalence", color="#3b82f6", alpha=0.85)
rects2 = ax.bar(x + width/2, df_per_class["Pred Count"], width, label="Predicted Prevalence", color="#10b981", alpha=0.85)

ax.set_ylabel("Number of Cases", fontsize=12)
ax.set_title("Pathology Distribution: Ground Truth vs. NaijaCXR-VLM Predictions", fontsize=14, fontweight="bold", pad=15)
ax.set_xticks(x)
ax.set_xticklabels(df_per_class["Pathology"], rotation=25, ha="right", fontsize=11)
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.5)

# Label bars with count values
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()


In [ ]:
# 2. Per-Pathology F1-Score & Diagnostic Sensitivity / Specificity
fig, ax = plt.subplots(figsize=(12, 5))

metrics_to_plot = df_per_class.set_index("Pathology")[["Precision", "Recall (Sensitivity)", "Specificity", "F1-Score"]]
metrics_to_plot.plot(kind="bar", ax=ax, colormap="viridis", width=0.75, alpha=0.9)

ax.set_ylim(0, 1.1)
ax.set_ylabel("Score (0.0 - 1.0)", fontsize=12)
ax.set_title("Diagnostic Performance Profile per Pathology", fontsize=14, fontweight="bold", pad=15)
ax.set_xticklabels(df_per_class["Pathology"], rotation=25, ha="right", fontsize=11)
ax.legend(loc="lower right", fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# 3. Multi-Label Confusion Matrices Grid
cols = 4
rows = int(np.ceil(num_classes / cols))
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.flatten()

for i, cls_name in enumerate(classes):
    cm = conf_matrices[i]
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=axes[i],
        xticklabels=["Neg", "Pos"],
        yticklabels=["Neg", "Pos"]
    )
    axes[i].set_title(f"{cls_name} (F1: {df_per_class.loc[i, 'F1-Score']:.2f})", fontsize=11, fontweight="bold")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Ground Truth")

# Hide extra subplots
for j in range(num_classes, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Clinical Pathology Confusion Matrices", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## 9. Qualitative Clinical Error Analysis
Inspect cases where the model disagreed with the ground truth radiologist findings.


In [ ]:
# Identify Discrepancies
df_raw["exact_match"] = [set(gt) == set(pr) for gt, pr in zip(y_true_labels, y_pred_labels)]

discrepant_cases = df_raw[~df_raw["exact_match"]].copy()
print(f"🔍 Found {len(discrepant_cases)} / {len(df_raw)} cases with diagnostic discrepancies.")

# Display sample error cases
sample_errors = discrepant_cases.sample(min(3, len(discrepant_cases)), random_state=42)

for i, (_, row) in enumerate(sample_errors.iterrows(), 1):
    print("=" * 80)
    print(f"Case {i} - Image: {row.get('image', 'N/A')}")
    print(f"🏷️ Ground Truth Labels: {row['gt_labels']}")
    print(f"🏷️ Predicted Labels:    {row['pred_labels']}")
    print("-" * 40)
    print(f"📋 Ground Truth Report:\n{row['ground_truth']}")
    print("-" * 40)
    print(f"🤖 Predicted Report:\n{row['prediction']}")
    print("=" * 80)
